### Step 1: Setting up communication:
### setup_sender: Creates a socket to send messages to the multicast group.
### setup_receiver: Listens for messages from the multicast group and processes them.

In [ ]:
import socket
import struct
import asyncio
import uuid

In [ ]:
MULTICAST_GROUP = '224.0.0.10'
MULTICAST_PORT = 55000

In [ ]:
# Function to set up a multicast socket for sending messages
def setup_sender():
    sender_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM, socket.IPPROTO_UDP)
    sender_socket.setsockopt(socket.IPPROTO_IP, socket.IP_MULTICAST_TTL, 2)
    return sender_socket



In [ ]:
# Function to set up a multicast socket for receiving messages
def setup_receiver():
    receiver_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM, socket.IPPROTO_UDP)
    receiver_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    receiver_socket.bind(('', MULTICAST_PORT))

    group = socket.inet_aton(MULTICAST_GROUP)
    mreq = struct.pack('4sL', group, socket.INADDR_ANY)
    receiver_socket.setsockopt(socket.IPPROTO_IP, socket.IP_ADD_MEMBERSHIP, mreq)
    return receiver_socket

### Step 2: Generate UUID 
### Each node in the system needs a unique identifier for operations like leader election and message tracking. UUID ensures no two nodes have the same ID

In [ ]:


def generate_node_id():
    return str(uuid.uuid4())

node_id = generate_node_id()
print(f"Node ID: {node_id}")

### Step 3: Listen for Messages (Non-blocking)
### to listen wihtout blocking other operations
### Asyncio: for non blocking 

In [ ]:
async def listen_for_messages(receiver_socket):
    while True:
        try:
            data, addr = receiver_socket.recvfrom(1024)
            print(f"Message from {addr}: {data.decode()}")
        except Exception as e:
            print(f"Error: {e}")
        await asyncio.sleep(0.1)  # Prevent blocking

### Step 4: Send Messages 
### Send Messages to annouce presence and participate in elections

In [ ]:
async def broadcast_message(sender_socket, message):
    sender_socket.sendto(message.encode(), (MULTICAST_GROUP, MULTICAST_PORT))
    print(f"Broadcasted: {message}")
    await asyncio.sleep(0.1)

### Step 5: Heartbeat Simulation Every 5 seconds 

In [ ]:
async def send_heartbeat(sender_socket):
    while True:
        await broadcast_message(sender_socket, f"HEARTBEAT from {node_id}")
        await asyncio.sleep(5)  # Send heartbeat every 5 seconds

## Main

In [ ]:
async def main():
    sender = setup_sender()
    receiver = setup_receiver()

    # Start listening for messages
    asyncio.create_task(listen_for_messages(receiver))

    # Simulate heartbeat from the leader
    asyncio.create_task(send_heartbeat(sender))

    # Simulate node joining
    await broadcast_message(sender, f"JOIN {node_id}")

# Run the main logic
await main()